# CALM-VAD — run on the paper's benchmarks (Colab)

This uses **pre-extracted HR pose data** (ShanghaiTech-HR, Avenue-HR) that the
pose-VAD community publishes *with* ground truth — so **no video downloads and
no GPU pose extraction**. Everything here is CPU-light.

Run the cells in order. Each dataset = repeat cells 3-6 with a new link + tag.

## Cell 1 — install

In [ ]:
!pip -q install numpy scipy scikit-learn pyyaml gdown
print('ok')

## Cell 2 — get the code + sanity check
Set `REPO_URL` to your GitHub repo, or leave `''` to upload a project zip.

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/FaizanAbbas512/Sentrix.git'
import os, sys
if REPO_URL:
    !rm -rf /content/sentrix && git clone --depth 1 $REPO_URL /content/sentrix
else:
    from google.colab import files
    print('upload a zip of the project (must contain calm/):')
    up = files.upload(); z = next(iter(up))
    !rm -rf /content/sentrix && mkdir -p /content/sentrix && unzip -q "$z" -d /content/sentrix
    s = [d for d in os.listdir('/content/sentrix') if os.path.isdir(f'/content/sentrix/{d}')]
    if 'calm' not in s and len(s) == 1: !cp -r /content/sentrix/{s[0]}/* /content/sentrix/
os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
assert os.path.isdir('calm'), 'calm/ not found'
!mkdir -p data/pose results
!python -m calm.selftest | tail -3

## Cell 3 — download the HR pose data (ShanghaiTech-HR)

Get a Google-Drive link for the extracted pose data + ground truth:
* **STG-NF** repo: https://github.com/orhir/STG-NF  → README → *Data Preparation*
  (it links Drive folders for `ShanghaiTech` and `Avenue`, each with
  `pose/train`, `pose/test`, `gt/test_frame_mask`).
* or **GEPC**: https://github.com/amirmk89/gepc  (ShanghaiTech pose).
* or **MoCoDAD**: https://github.com/aleflabo/MoCoDAD  → `download_data`.

Paste the Drive **file id** or **share link** below. `gdown --folder` also works
for a shared folder link.

In [ ]:
DATA_URL = ''   # Drive file id OR full share link (a .zip/.tar of the dataset's pose+gt)
IS_FOLDER = False   # True if DATA_URL is a shared *folder* link

import gdown, os, glob, zipfile, tarfile
os.makedirs('/content/hrdata', exist_ok=True)
if IS_FOLDER:
    gdown.download_folder(DATA_URL, output='/content/hrdata', quiet=False, use_cookies=False)
else:
    src = DATA_URL if 'http' in DATA_URL else f'https://drive.google.com/uc?id={DATA_URL}'
    gdown.download(src, '/content/hrdata/data.arc', quiet=False, fuzzy=True)
    a = '/content/hrdata/data.arc'
    if zipfile.is_zipfile(a): zipfile.ZipFile(a).extractall('/content/hrdata')
    elif tarfile.is_tarfile(a): tarfile.open(a).extractall('/content/hrdata')

# show what we got
for d in sorted({os.path.dirname(p) for p in glob.glob('/content/hrdata/**/*', recursive=True)})[:40]:
    print(d)

## Cell 4 — find the pose/gt folders and run the harness

Auto-detects a `pose/test` (or `test`) folder of per-clip JSONs and a
`gt/test_frame_mask` (or `*frame_mask*`) folder of `.npy` masks, then runs the
full 5-axis evaluation.

In [ ]:
TAG = 'shanghaitech'   # <- name this run (shanghaitech / avenue / ...)
import glob, os

def find_dir(patterns, must_have_ext):
    for pat in patterns:
        for d in glob.glob(f'/content/hrdata/**/{pat}', recursive=True):
            if os.path.isdir(d) and glob.glob(f'{d}/*{must_have_ext}'):
                return d
    return None

pose_dir = find_dir(['**/pose/test', '**/pose/testing', '**/test', '**/testing'], '.json')
gt_dir   = find_dir(['**/gt/test_frame_mask', '**/*frame_mask*', '**/*test*mask*', '**/gt'], '.npy')
assert pose_dir, 'no folder of per-clip *.json found under /content/hrdata (check Cell 3 output)'
gt_dir = gt_dir or pose_dir     # harness needs 2 args; a wrong gt_dir just yields 0 events
print('pose_dir =', pose_dir)
print('gt_dir   =', gt_dir)

!python -m calm.harness --shanghaitech "$pose_dir" "$gt_dir" --tag $TAG

## Cell 5 — read the report

In [ ]:
import json
r = json.load(open(f'results/calm_report_{TAG}.json'))
print('=== ', TAG, ' ===')
for s in r['streams']:
    faph = {k.replace('faph@recall=',''): v for k, v in s.items() if k.startswith('faph@')}
    print(f"  {s['label']:<26} AUC {s['frame_auc']:.3f}  eventF1 {s['event']['f1_avg']:.3f}  FAPH {faph}")
print('  calibration ECE raw->cal :', round(r['calibration']['ece_raw'], 4), '->', round(r['calibration']['ece_cal'], 4))
print('  M4 budgets:')
for b, x in r['risk_control_M4']['by_budget'].items():
    print(f"     {b}: tau={x['tau']:.3f}  certified={x['certified']}  held-out FAPH={x['held_out_normal_faph']}")
print('  decision-layer ms/frame :', round(r['cost']['decision_layer_ms_per_frame_mean'], 3))

## Cell 6 — download this run's results

In [ ]:
!zip -qr /content/calm_$TAG.zip data/pose/*$TAG* results/*$TAG*
from google.colab import files; files.download(f'/content/calm_{TAG}.zip')

## Cell 7 — next dataset / cross-dataset

**Avenue-HR:** re-run Cells 3-6 with the Avenue Drive link and `TAG='avenue'`
(clear `/content/hrdata` first: `!rm -rf /content/hrdata`).

**Cross-dataset drop** (fit on A, test on B) — after you have two pose JSONs:

In [ ]:
# build generic JSONs first (one per dataset) with calm.datasets, then:
import json
def merge(fit_json, test_json, out):
    a = json.load(open(fit_json)); b = json.load(open(test_json))
    for x in a['clips']: x['split'] = 'calib'
    for x in b['clips']: x['split'] = 'test'
    json.dump({'fps': a['fps'], 'clips': a['clips'] + b['clips']}, open(out, 'w'))
# to get a generic JSON from an HR folder:
# from calm.datasets import load_shanghaitech_hr
# import json, dataclasses
# clips = load_shanghaitech_hr(pose_dir, gt_dir)
# json.dump({'fps': 24, 'clips': [dataclasses.asdict(c) for c in clips]}, open('data/pose/shanghaitech.json','w'))

## Cell 8 (alternative) — only have RAW videos? extract poses on GPU

Use this **instead of Cells 3-4** if you have a folder of video files + a
folder of per-clip GT (`.npy` masks / `.txt` intervals / `.json`).
Needs `Runtime -> T4 GPU` and `!pip -q install ultralytics opencv-python-headless`.

In [ ]:
TAG = 'myset'
!pip -q install ultralytics opencv-python-headless
!python -m calm.extract_poses --videos /content/data/videos --gt /content/data/gt \
    --out data/pose/$TAG.json --split test --weights yolo11n-pose.pt --imgsz 640 --device 0
!python -m calm.harness --generic data/pose/$TAG.json --tag $TAG